# Graph Neural Network fingerprints for Data 4.3

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('data_4.csv')
df_cdd = df_cdd.rename(columns={
    "GU CRC L6: Average EC50": "gu_ec50",
    "GU CRC L6: Average EC50 Standard Deviation (±)": "gu_std",
    "GU CRC L6: Average EC50 Count": "gu_count",
    "GU CRC L6: Average Emax": "gu_emax",
    "B2 cAMP CRC L6: Average Emax Standard Deviation (±)": "gu_emax_std",
    "B2 cAMP CRC L6: Average Emax Count": "gu_emax_count",
    "B2 cAMP CRC L6: Average EC50 2": "camp_ec50",
    'B2 cAMP CRC L6: Average EC50 2 Standard Deviation (±)': "camp_std",
    'B2 cAMP CRC L6: Average EC50 2 Count': "camp_count",
    "B2 cAMP CRC L6: Average Emax": "camp_emax",
    "GU CRC L6: Average Emax Standard Deviation (±)": "camp_emax_std",
    "GU CRC L6: Average Emax Count": "camp_emax_count"
})

Calculates the maximum value from the CRC doses for glucose uptake and cAMP

In [ ]:
camp_cols = [
    "B2 cAMP CRC L6: cAMP (-5) (of iso)",
    "B2 cAMP CRC L6: cAMP (-6) (of iso)",
    "B2 cAMP CRC L6: cAMP (-7) (of iso)",
    "B2 cAMP CRC L6: cAMP (-8) (of iso)",
    "B2 cAMP CRC L6: cAMP (-9) (of iso)",
    "B2 cAMP CRC L6: cAMP (-10) (of iso)"
]

gu_cols = [
    "GU CRC L6: GU (-5) (of iso)",
    "GU CRC L6: GU (-6) (of iso)",
    "GU CRC L6: GU (-7) (of iso)",
    "GU CRC L6: GU (-8) (of iso)",
    "GU CRC L6: GU (-9) (of iso)",
    "GU CRC L6: GU (-10) (of iso)"
]

df_cdd["camp_emax"] = df_cdd[camp_cols].max(axis=1, skipna=True)
df_cdd["gu_emax"] = df_cdd[gu_cols].max(axis=1, skipna=True)

Drops the columns that will not be needed for the ML models and Explainable methods

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'B2 cAMP CRC L6: Basal (of iso)',
                      'B2 cAMP CRC L6: cAMP Clen (of iso)',
                      "B2 cAMP CRC L6: cAMP (-5) (of iso)",
                      'B2 cAMP CRC L6: cAMP (-6) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-7) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-8) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-9) (of iso)',
                      'B2 cAMP CRC L6: cAMP (-10) (of iso)',
                      'B2 cAMP CRC L6: Top (of iso)',
                      'B2 cAMP CRC L6: Bottom (of iso)',
                      'B2 cAMP CRC L6: Average EC50 batch',
                      'B2 cAMP CRC L6: Average EC50 batch Standard Deviation (±)',
                      'B2 cAMP CRC L6: Average EC50 batch Count',
                      'B2 cAMP CRC L6: Emax',
                      'GU CRC L6: GU (-5) (of iso)',
                      'GU CRC L6: GU (-6) (of iso)',
                      'GU CRC L6: GU (-7) (of iso)',
                      'GU CRC L6: GU (-8) (of iso)',
                      'GU CRC L6: GU (-9) (of iso)',
                      'GU CRC L6: GU (-10) (of iso)',
                      'GU CRC L6: Bottom L6G (of iso)',
                      'GU CRC L6: Top L6G (of iso)',
                      'GU CRC L6: Average EC50 batch',
                      'GU CRC L6: Average EC50 batch Standard Deviation (±)',
                      'GU CRC L6: Average EC50 batch Count',
                      'GU CRC L6: Emax'
                    ], axis=1)


In [ ]:
import numpy as np

# Group by Molecule Name and SMILES, then aggregate
df_cdd = df_cdd.groupby(['Molecule Name', 'SMILES']).agg({
    # For EC50 values: take mean
    'camp_ec50': 'mean',
    'gu_ec50': 'mean',
    
    # For Emax values: take mean
    'camp_emax': 'mean',
    'gu_emax': 'mean',
    
    # For standard deviations: pool them using root mean square
    'camp_std': lambda x: np.sqrt(np.mean(x**2)),
    'gu_std': lambda x: np.sqrt(np.mean(x**2)),
    'gu_emax_std': lambda x: np.sqrt(np.mean(x**2)),
    'camp_emax_std': lambda x: np.sqrt(np.mean(x**2)),
}).reset_index()

In [ ]:
df_cdd = df_cdd.drop(['camp_std',
                      'gu_emax_std',
                      'gu_std',
                      'camp_emax_std'
                    ], axis=1)

df_cdd = df_cdd[["Molecule Name", "SMILES", "gu_ec50", "gu_emax", "camp_ec50", "camp_emax"]]

df_cdd = df_cdd.dropna(subset=[ 'gu_ec50',
                                'camp_ec50' ])

In [ ]:
df = df_cdd
df.columns = df.columns.str.strip()

# Basic sanity check
TARGET_NAMES = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']
required = {'Molecule Name', 'SMILES'} | set(TARGET_NAMES)
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

print(f'Dataset shape: {df.shape}')
print(df[TARGET_NAMES].describe())
df.head()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D

# Sklearn
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# PyTorch Geometric
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import NNConv, Set2Set
from torch_geometric.explain import Explainer, GNNExplainer

# Captum (Integrated Gradients)
try:
    from captum.attr import IntegratedGradients
    CAPTUM_OK = True
    print('Captum available — Integrated Gradients enabled.')
except ImportError:
    print('Captum not installed — Integrated Gradients will be skipped.')
    CAPTUM_OK = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports successful.')


## 2. Molecular Graph Featurisation

Each molecule is converted to a **PyTorch Geometric `Data` object** with:

- **Node features (atom):** atomic number (one-hot, 10 types), degree, formal charge, radical electrons, hybridisation (one-hot, 5 types), aromaticity, H-count -> **20-dimensional** vector per atom.
- **Edge features (bond):** bond type (one-hot: SINGLE, DOUBLE, TRIPLE, AROMATIC), conjugation, ring membership -> **6-dimensional** per bond.
- **y:** 4-element target vector `[gu_ec50, gu_emax, camp_ec50, camp_emax]`.
- **w:** uniform sample weight (set to 1.0 for all molecules — standard deviation not considered).

In [ ]:
ATOM_TYPES = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]   # H C N O F P S Cl Br I
HYBRIDISATION = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

def one_hot(val, choices):
    return [int(val == c) for c in choices]

def atom_features(atom):
    return (
        one_hot(atom.GetAtomicNum(), ATOM_TYPES)           # 10
        + [atom.GetDegree() / 6.0]                         #  1
        + [atom.GetFormalCharge() / 4.0]                   #  1
        + [atom.GetNumRadicalElectrons()]                   #  1
        + one_hot(atom.GetHybridization(), HYBRIDISATION)  #  5
        + [int(atom.GetIsAromatic())]                      #  1
        + [atom.GetTotalNumHs() / 4.0]                     #  1
    )                                                       # = 20 dims

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def bond_features(bond):
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)  # 4
        + [int(bond.GetIsConjugated())]           # 1
        + [int(bond.IsInRing())]                  # 1
    )                                             # = 6 dims

NODE_DIM = 20
EDGE_DIM =  6

def mol_to_graph(smi, y_vals, w_scalar):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]
        edge_attr  = [[0] * EDGE_DIM]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)
    y = torch.tensor(y_vals, dtype=torch.float).unsqueeze(0)
    w = torch.tensor([w_scalar], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, w=w)


TARGET_NAMES = ['gu_ec50', 'gu_emax', 'camp_ec50', 'camp_emax']
y_all = df[TARGET_NAMES].values
# Uniform weights: standard deviation not considered
w_all = np.ones(len(df))

graph_list = []
for idx, (_, row) in enumerate(df.iterrows()):  # FIX: use enumerate for positional indexing
    g = mol_to_graph(row['SMILES'], y_all[idx], float(w_all[idx]))
    if g is not None:
        graph_list.append(g)

print(f'Graphs built: {len(graph_list)}')
print(f'Node feature dim: {NODE_DIM}   Edge feature dim: {EDGE_DIM}')
print(f'Example graph: {graph_list[0]}')

### 2.1 Message Passing Neural Network Architecture

The MPNN uses **NNConv** (edge-conditioned convolutions) to propagate atom-level information along bonds, weighted by learnable edge embeddings. **Set2Set** pooling provides an order-invariant graph-level readout, followed by an MLP predicting all 4 targets jointly.

```
Input atoms (NODE_DIM=20) + bonds (EDGE_DIM=6)
         |
  Edge Embedding Net  ->  NNConv message kernel
         |
  NNConv x n_layers  +  BatchNorm  +  ReLU  +  Dropout
         |
  Set2Set pooling  (graph-level readout, 2 x hidden_dim)
         |
  MLP Head  ->  [gu_ec50, gu_emax, camp_ec50, camp_emax]
```

**Loss:** Weighted MSE, averaged over all 4 targets.

In [ ]:
class MPNNModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, edge_dim=EDGE_DIM,
                 hidden_dim=128, n_layers=3, n_outputs=4, dropout=0.2):
        super().__init__()
        self.n_layers = n_layers
        self.atom_proj = nn.Sequential(nn.Linear(node_dim, hidden_dim), nn.ReLU())
        self.convs = nn.ModuleList()
        for _ in range(n_layers):
            edge_net = nn.Sequential(
                nn.Linear(edge_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim * hidden_dim),
            )
            self.convs.append(NNConv(hidden_dim, hidden_dim, edge_net, aggr='mean'))
        self.bn = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(n_layers)])
        self.readout = Set2Set(hidden_dim, processing_steps=3)
        self.dropout_layer = nn.Dropout(dropout)
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_outputs),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.atom_proj(x)
        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index, edge_attr)
            h = self.bn[i](h)
            h = F.relu(h)
            h = self.dropout_layer(h)
        out = self.readout(h, batch)   # (B, 2*hidden_dim)
        return self.mlp(out)           # (B, 4)


def weighted_mse(pred, target, weight):
    loss_per_sample = ((pred - target) ** 2).mean(dim=1)
    return (loss_per_sample * weight).sum() / weight.sum()


print('MPNN architecture defined.')
_m = MPNNModel().to(DEVICE)
print(_m)
del _m


## 3. Cross-Validation Setup

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(graph_list))


def cv_metrics_np(y_true, y_pred, weights=None, target_names=TARGET_NAMES):
    results = {}
    for i, name in enumerate(target_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        w  = weights if weights is not None else np.ones(len(yt))
        w  = w / w.sum() * len(w)
        rmse = np.sqrt(np.average((yt - yp) ** 2, weights=w))
        mae  = np.average(np.abs(yt - yp), weights=w)
        ss_res = np.sum(w * (yt - yp) ** 2)
        ss_tot = np.sum(w * (yt - np.average(yt, weights=w)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    return results


def print_cv_summary(all_fold_metrics, model_name):
    print(f'\n{chr(8212)*65}')
    print(f'  {model_name} -- 5-Fold CV Summary (weighted)')
    print(f'{chr(8212)*65}')
    for target in TARGET_NAMES:
        rmse_vals = [m[target]['RMSE'] for m in all_fold_metrics]
        mae_vals  = [m[target]['MAE']  for m in all_fold_metrics]
        r2_vals   = [m[target]['R2']   for m in all_fold_metrics]
        print(f'  [{target.upper()}]  '
              f'RMSE: {np.mean(rmse_vals):.4f}+/-{np.std(rmse_vals):.4f} | '
              f'MAE: {np.mean(mae_vals):.4f}+/-{np.std(mae_vals):.4f} | '
              f'R2: {np.mean(r2_vals):.4f}+/-{np.std(r2_vals):.4f}')
    print(f'{chr(8212)*65}')


print('CV helpers defined.')


## 4. MPNN Training -- 5-Fold Cross-Validation

Each fold trains the MPNN for up to `MAX_EPOCHS` with early stopping on validation weighted MSE.

In [ ]:
HIDDEN_DIM = 128
N_LAYERS   = 3
DROPOUT    = 0.2
BATCH_SIZE = 32
MAX_EPOCHS = 150
LR         = 1e-3
PATIENCE   = 20

mpnn_fold_metrics = []
mpnn_fold_models  = []  # (model, val_idx, y_true_val, y_pred_val, w_val)

for fold, (train_idx, val_idx) in enumerate(kf.split(indices)):
    print(f'\n-- Fold {fold+1}/5 --')
    train_graphs = [graph_list[i] for i in train_idx]
    val_graphs   = [graph_list[i] for i in val_idx]
    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE, shuffle=False)

    model     = MPNNModel(hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, dropout=DROPOUT).to(DEVICE)
    optimiser = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimiser, patience=10, factor=0.5)

    best_val_loss = float('inf')
    patience_cnt  = 0
    best_state    = None

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimiser.zero_grad()
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            loss = weighted_mse(pred, batch.y, batch.w)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            train_loss += loss.item() * batch.num_graphs
        train_loss /= len(train_graphs)

        model.eval()
        val_loss = 0.0
        y_true_list, y_pred_list, w_list = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                val_loss += weighted_mse(pred, batch.y, batch.w).item() * batch.num_graphs
                y_true_list.append(batch.y.cpu().numpy())
                y_pred_list.append(pred.cpu().numpy())
                w_list.append(batch.w.cpu().numpy())
        val_loss /= len(val_graphs)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = deepcopy(model.state_dict())
            patience_cnt  = 0
        else:
            patience_cnt += 1

        if epoch % 25 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d} | train={train_loss:.5f} | val={val_loss:.5f}')

        if patience_cnt >= PATIENCE:
            print(f'  Early stop at epoch {epoch}.')
            break

    model.load_state_dict(best_state)
    model.eval()
    y_true_all = np.vstack(y_true_list)
    y_pred_all = np.vstack(y_pred_list)
    w_val_all  = np.concatenate(w_list)

    metrics = cv_metrics_np(y_true_all, y_pred_all, weights=w_val_all)
    mpnn_fold_metrics.append(metrics)
    mpnn_fold_models.append((deepcopy(model), val_idx, y_true_all, y_pred_all, w_val_all))

    rmse_str = '  '.join(f'{t}={metrics[t]["RMSE"]:.4f}' for t in TARGET_NAMES)
    print(f'  Best val_loss={best_val_loss:.5f} | {rmse_str}')

print_cv_summary(mpnn_fold_metrics, 'MPNN')


## 5. Model Performance

In [ ]:
rows = []
for target in TARGET_NAMES:
    for metric in ['RMSE', 'MAE', 'R2']:
        vals = [m[target][metric] for m in mpnn_fold_metrics]
        rows.append({'Target': target, 'Metric': metric,
                     'Mean': np.mean(vals), 'Std': np.std(vals)})
results_df = pd.DataFrame(rows)
pivot = results_df.pivot_table(index='Target', columns='Metric', values=['Mean', 'Std'])
print(pivot.round(4))

best_fold_idx = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in mpnn_fold_metrics
])
_, _, y_true_best, y_pred_best, _ = mpnn_fold_models[best_fold_idx]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
colors = ['steelblue', 'teal', 'darkorange', 'firebrick']
for ax, col, name in zip(axes.flat, colors, TARGET_NAMES):
    ti = TARGET_NAMES.index(name)
    ax.scatter(y_true_best[:, ti], y_pred_best[:, ti],
               alpha=0.6, color=col, edgecolors='white', s=40)
    mn = min(y_true_best[:, ti].min(), y_pred_best[:, ti].min())
    mx = max(y_true_best[:, ti].max(), y_pred_best[:, ti].max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name.upper()}  RMSE={mpnn_fold_metrics[best_fold_idx][name]["RMSE"]:.4f}')
plt.suptitle('MPNN: Predicted vs Actual (Best Fold)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Explainable AI

### 6.1 Bias Score Helper

The **bias score** for a graph element e is:

    BiasScore(e) = (phi_gu_ec50(e) - phi_camp_ec50(e)) + (phi_gu_emax(e) - phi_camp_emax(e))

A **positive** score means the element promotes the desired functional selectivity profile (high GU, low cAMP).

In [ ]:
def bias_score_plot_atoms(attr_dict, feature_names, top_n=20, title='Bias Score'):
    mean = {t: np.asarray(attr_dict[t]) for t in TARGET_NAMES}
    ec50_bias = mean['gu_ec50'] - mean['camp_ec50']
    emax_bias = mean['gu_emax'] - mean['camp_emax']
    bias      = ec50_bias + emax_bias
    n = min(top_n, len(feature_names))
    idx = np.argsort(np.abs(bias))[::-1][:n]
    top_names = [feature_names[i] for i in idx]
    top_bias  = bias[idx]
    colors = ['#2196F3' if b > 0 else '#F44336' for b in top_bias]
    plt.figure(figsize=(10, 6))
    plt.barh(top_names[::-1], top_bias[::-1], color=colors[::-1])
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Bias Score  (up = High GU EC50/Emax & Low cAMP EC50/Emax)')
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return pd.DataFrame({
        'Feature':        top_names,
        'BiasScore':      top_bias,
        'EC50_bias':      ec50_bias[idx],
        'Emax_bias':      emax_bias[idx],
        'Mean_gu_ec50':   mean['gu_ec50'][idx],
        'Mean_gu_emax':   mean['gu_emax'][idx],
        'Mean_camp_ec50': mean['camp_ec50'][idx],
        'Mean_camp_emax': mean['camp_emax'][idx],
    })


print('Bias-score helper defined.')


### 6.2 GNNExplainer -- Node & Edge Mask Attribution

**GNNExplainer** learns soft masks over edges and node features that maximise mutual information between the masked sub-graph and the model prediction. We run it for each of the 4 targets on the first `MAX_EXPLAIN` validation molecules, then aggregate importances by **atom type**.

In [ ]:
best_fold_idx = np.argmin([
    np.mean([m[t]['RMSE'] for t in TARGET_NAMES]) for m in mpnn_fold_metrics
])
best_model, val_idx_best, _, _, _ = mpnn_fold_models[best_fold_idx]
best_model.eval()
val_graphs_best = [graph_list[i] for i in val_idx_best]
print(f'Using fold {best_fold_idx+1} ({len(val_graphs_best)} validation molecules)')


class SingleOutputWrapper(nn.Module):
    def __init__(self, mpnn, target_idx):
        super().__init__()
        self.mpnn = mpnn
        self.target_idx = target_idx

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        out = self.mpnn(x, edge_index, edge_attr, batch)
        return out[:, self.target_idx].unsqueeze(1)


ATOM_SYMBOLS = ['H', 'C', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Br', 'I', 'Other']
MAX_EXPLAIN  = min(20, len(val_graphs_best))
gnn_atom_attr = {t: [] for t in TARGET_NAMES}

print(f'Running GNNExplainer on first {MAX_EXPLAIN} validation molecules ...')

for mol_idx in range(MAX_EXPLAIN):
    g   = val_graphs_best[mol_idx].to(DEVICE)
    bat = torch.zeros(g.x.size(0), dtype=torch.long, device=DEVICE)

    for t_idx, t_name in enumerate(TARGET_NAMES):
        wrapper = SingleOutputWrapper(best_model, t_idx).to(DEVICE)
        wrapper.eval()

        explainer = Explainer(
            model            = wrapper,
            algorithm        = GNNExplainer(epochs=200),
            explanation_type = 'model',
            node_mask_type   = 'attributes',
            edge_mask_type   = 'object',
            model_config     = dict(mode='regression', task_level='graph', return_type='raw'),
        )
        explanation = explainer(
            x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr, batch=bat
        )

        if explanation.node_mask is not None:
            node_imp = explanation.node_mask.abs().mean(dim=1).detach().cpu().numpy()
        else:
            edge_mask = explanation.edge_mask.detach().cpu().numpy()
            ei        = g.edge_index.cpu().numpy()
            node_imp  = np.zeros(g.num_nodes)
            for k, (src, _) in enumerate(ei.T):
                node_imp[src] += edge_mask[k]
            node_imp /= (node_imp.max() + 1e-8)

        gnn_atom_attr[t_name].append(node_imp)

    if (mol_idx + 1) % 5 == 0:
        print(f'  {mol_idx+1}/{MAX_EXPLAIN} molecules done')

print('GNNExplainer complete.')


In [ ]:
def aggregate_by_atom_type(attr_lists, val_idx, n_mol):
    atom_sum   = {s: {t: 0.0 for t in TARGET_NAMES} for s in ATOM_SYMBOLS}
    atom_count = {s: 0 for s in ATOM_SYMBOLS}
    for mol_idx in range(n_mol):
        smi = df['SMILES'].iloc[val_idx[mol_idx]]
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        n_atoms = mol.GetNumAtoms()
        for t_name in TARGET_NAMES:
            if mol_idx >= len(attr_lists[t_name]):
                continue
            imp = attr_lists[t_name][mol_idx]
            for a_idx in range(min(n_atoms, len(imp))):
                sym = mol.GetAtomWithIdx(a_idx).GetSymbol()
                sym = sym if sym in ATOM_SYMBOLS else 'Other'
                atom_sum[sym][t_name] += imp[a_idx]
                if t_name == TARGET_NAMES[0]:
                    atom_count[sym] += 1
    result = {}
    for sym in ATOM_SYMBOLS:
        cnt = max(atom_count[sym], 1)
        result[sym] = {t: atom_sum[sym][t] / cnt for t in TARGET_NAMES}
    return result


atom_agg = aggregate_by_atom_type(gnn_atom_attr, val_idx_best, MAX_EXPLAIN)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors_t = ['steelblue', 'teal', 'darkorange', 'firebrick']
for ax, t_name, col in zip(axes.flat, TARGET_NAMES, colors_t):
    vals = [atom_agg[sym][t_name] for sym in ATOM_SYMBOLS]
    ax.bar(ATOM_SYMBOLS, vals, color=col, alpha=0.85)
    ax.set_title(f'GNNExplainer Atom Attribution -- {t_name.upper()}')
    ax.set_xlabel('Atom type')
    ax.set_ylabel('Mean node importance')
plt.suptitle('GNNExplainer: Atom-Type Attribution per Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

atom_bias_dict = {t: np.array([atom_agg[sym][t] for sym in ATOM_SYMBOLS]) for t in TARGET_NAMES}
gnn_bias_df = bias_score_plot_atoms(
    atom_bias_dict, ATOM_SYMBOLS, top_n=len(ATOM_SYMBOLS),
    title='GNNExplainer Atom-Type Bias Score (High GU EC50/Emax & Low cAMP EC50/Emax)'
)
display(gnn_bias_df)


### 6.3 Visualise GNNExplainer Attribution on a Representative Molecule

The first explained validation molecule is drawn with atoms coloured by mean GNNExplainer node importance averaged over all 4 targets (white = low, red = high importance).


In [ ]:
from PIL import Image
from io import BytesIO


def draw_mol_importance(smi, node_imp_per_target, target_names=TARGET_NAMES):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print('Invalid SMILES.')
        return
    AllChem.Compute2DCoords(mol)
    n = mol.GetNumAtoms()
    all_imp  = np.stack([node_imp_per_target[t][:n] for t in target_names], axis=0)
    mean_imp = all_imp.mean(axis=0)
    norm_imp = (mean_imp - mean_imp.min()) / (mean_imp.max() - mean_imp.min() + 1e-8)
    highlight_atoms  = list(range(n))
    highlight_colors = {i: (1.0, 1.0 - float(norm_imp[i]), 1.0 - float(norm_imp[i]))
                        for i in range(n)}
    drawer = rdMolDraw2D.MolDraw2DCairo(400, 300)
    drawer.drawOptions().addAtomIndices = False
    rdMolDraw2D.PrepareMolForDrawing(mol)
    drawer.DrawMolecule(mol,
                        highlightAtoms=highlight_atoms,
                        highlightAtomColors=highlight_colors,
                        highlightBonds=[],
                        highlightBondColors={})
    drawer.FinishDrawing()
    bio = BytesIO(drawer.GetDrawingText())
    img = Image.open(bio)
    plt.figure(figsize=(5, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'GNNExplainer Atom Importance (avg 4 targets)\n{smi[:60]}', fontsize=9)
    plt.tight_layout()
    plt.show()


ex_mol_idx = 0
ex_smi     = df['SMILES'].iloc[val_idx_best[ex_mol_idx]]
ex_attr    = {t: gnn_atom_attr[t][ex_mol_idx] for t in TARGET_NAMES}
print(f'Visualising: {ex_smi}')
try:
    draw_mol_importance(ex_smi, ex_attr)
except Exception as e:
    print(f'Visualisation error: {e}')
    mol = Chem.MolFromSmiles(ex_smi)
    AllChem.Compute2DCoords(mol)
    img = Draw.MolToImage(mol, size=(400, 300))
    plt.imshow(img); plt.axis('off'); plt.title(ex_smi[:60]); plt.show()


### 6.4 Integrated Gradients -- Node-Feature Attribution

**Integrated Gradients** (Sundararajan et al., 2017) attributes model output to input node features by integrating gradients along a straight-line path from a zero baseline to the actual atom features. This reveals which atom descriptors (atom type, hybridisation, aromaticity, etc.) drive each target prediction.

The (n_atoms x node_dim) attribution matrix is averaged over atoms to obtain a per-feature summary, then averaged across all explained molecules.

In [ ]:
if not CAPTUM_OK:
    print('Captum not available -- skipping Integrated Gradients. Run: pip install captum')
else:
    NODE_FEATURE_NAMES = (
        [f'AtomType_{s}' for s in ['H','C','N','O','F','P','S','Cl','Br','I']]
        + ['Degree', 'FormalCharge', 'RadicalElectrons']
        + ['Hybrid_SP','Hybrid_SP2','Hybrid_SP3','Hybrid_SP3D','Hybrid_SP3D2']
        + ['IsAromatic', 'NumHs']
    )  # = 20

    class NodeFeatureWrapper(nn.Module):
        def __init__(self, mpnn, edge_index, edge_attr, batch, target_idx):
            super().__init__()
            self.mpnn       = mpnn
            self.edge_index = edge_index
            self.edge_attr  = edge_attr
            self.batch      = batch
            self.target_idx = target_idx

        def forward(self, x):
            """
            Captum passes x as (n_steps, num_nodes, num_features) during
            batched attribution. We must replicate the graph structure for
            every step so the GNN message-passing sees consistent sizes.
            """
            if x.dim() == 3:
                n_steps, num_nodes, num_feats = x.shape
                x_flat = x.view(n_steps * num_nodes, num_feats)

                # Offset edge indices for each copy of the graph
                edge_index_rep = torch.cat(
                    [self.edge_index + i * num_nodes for i in range(n_steps)],
                    dim=1
                )
                # Repeat edge attributes for each copy
                edge_attr_rep = self.edge_attr.repeat(n_steps, 1)

                # Batch vector: node i belongs to graph (i // num_nodes)
                batch_rep = torch.arange(n_steps, device=x.device).repeat_interleave(num_nodes)

                out = self.mpnn(x_flat, edge_index_rep, edge_attr_rep, batch_rep)
                # out shape: (n_steps, num_targets) — pick our target
                return out[:, self.target_idx].unsqueeze(1)

            else:
                # Single forward pass (num_nodes, num_features)
                out = self.mpnn(x, self.edge_index, self.edge_attr, self.batch)
                return out[:, self.target_idx].unsqueeze(1)

    ig_node_attr = {t: [] for t in TARGET_NAMES}
    MAX_IG = min(20, len(val_graphs_best))
    print(f'Running Integrated Gradients on first {MAX_IG} validation molecules ...')

    for mol_idx in range(MAX_IG):
        g        = val_graphs_best[mol_idx].to(DEVICE)
        ei       = g.edge_index
        ea       = g.edge_attr
        batch    = torch.zeros(g.x.size(0), dtype=torch.long, device=DEVICE)
        baseline = torch.zeros_like(g.x)

        for t_idx, t_name in enumerate(TARGET_NAMES):
            wrapper = NodeFeatureWrapper(best_model, ei, ea, batch, t_idx).to(DEVICE)
            wrapper.eval()
            ig = IntegratedGradients(wrapper)
            attr = ig.attribute(
                g.x.unsqueeze(0),          # (1, num_nodes, num_features)
                baselines=baseline.unsqueeze(0),
                n_steps=50,
                internal_batch_size=50,    # pass all steps at once → consistent shape
                return_convergence_delta=False,
            )
            node_mean = attr.squeeze(0).abs().mean(dim=0).detach().cpu().numpy()
            ig_node_attr[t_name].append(node_mean)

        if (mol_idx + 1) % 5 == 0:
            print(f'  {mol_idx+1}/{MAX_IG} molecules done')

    print('Integrated Gradients complete.')

    ig_mean = {t: np.mean(ig_node_attr[t], axis=0) for t in TARGET_NAMES}

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    colors_t = ['steelblue', 'teal', 'darkorange', 'firebrick']
    for ax, t_name, col in zip(axes.flat, TARGET_NAMES, colors_t):
        vals    = ig_mean[t_name]
        top_idx = np.argsort(vals)[::-1][:15]
        ax.barh([NODE_FEATURE_NAMES[i] for i in top_idx][::-1],
                 vals[top_idx][::-1], color=col, alpha=0.85)
        ax.set_title(f'IG Node Attribution -- {t_name.upper()}')
        ax.set_xlabel('Mean |IG| attribution')
    plt.suptitle('Integrated Gradients: Node Feature Attribution per Target',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    ig_bias_df = bias_score_plot_atoms(
        ig_mean, NODE_FEATURE_NAMES, top_n=15,
        title='IG Node-Feature Bias Score (High GU EC50/Emax & Low cAMP EC50/Emax)'
    )
    display(ig_bias_df)

## 7. Consensus Bias Score -- GNNExplainer & Integrated Gradients

Atom-type bias from **GNNExplainer** and node-feature bias from **Integrated Gradients** are presented side-by-side, revealing a multi-level picture of functional selectivity drivers.

In [ ]:
bias_gnn = (atom_bias_dict['gu_ec50'] - atom_bias_dict['camp_ec50']
          + atom_bias_dict['gu_emax'] - atom_bias_dict['camp_emax'])

if CAPTUM_OK:
    bias_ig_arr = np.array([ig_mean[t] for t in TARGET_NAMES])
    bias_ig = (bias_ig_arr[0] - bias_ig_arr[2]) + (bias_ig_arr[1] - bias_ig_arr[3])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors_gnn = ['#1976D2' if v > 0 else '#D32F2F' for v in bias_gnn]
    axes[0].barh(ATOM_SYMBOLS[::-1], bias_gnn[::-1], color=colors_gnn[::-1])
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('GNNExplainer: Atom-Type Bias Score')
    axes[0].set_xlabel('Bias Score')

    top15 = np.argsort(np.abs(bias_ig))[::-1][:15]
    colors_ig = ['#1976D2' if v > 0 else '#D32F2F' for v in bias_ig[top15]]
    axes[1].barh([NODE_FEATURE_NAMES[i] for i in top15][::-1],
                  bias_ig[top15][::-1], color=colors_ig[::-1])
    axes[1].axvline(0, color='black', lw=0.8)
    axes[1].set_title('Integrated Gradients: Node-Feature Bias Score')
    axes[1].set_xlabel('Bias Score')

    plt.suptitle('Consensus XAI Bias (GNNExplainer + Integrated Gradients)\n'
                 'up = High GU EC50/Emax   /   Low cAMP EC50/Emax',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\nAtom-type bias (GNNExplainer):')
    display(pd.DataFrame({'AtomType': ATOM_SYMBOLS, 'GNN_Bias': bias_gnn})
              .sort_values('GNN_Bias', ascending=False).reset_index(drop=True))

    print('\nTop node features by IG bias:')
    display(pd.DataFrame({'Feature': NODE_FEATURE_NAMES, 'IG_Bias': bias_ig})
              .sort_values('IG_Bias', ascending=False).head(10).reset_index(drop=True))

else:
    colors_gnn = ['#1976D2' if v > 0 else '#D32F2F' for v in bias_gnn]
    plt.figure(figsize=(8, 5))
    plt.barh(ATOM_SYMBOLS[::-1], bias_gnn[::-1], color=colors_gnn[::-1])
    plt.axvline(0, color='black', lw=0.8)
    plt.title('GNNExplainer Atom-Type Bias Score')
    plt.xlabel('Bias Score (up = High GU / Low cAMP)')
    plt.tight_layout()
    plt.show()
